# Notebook 07: Tensor Parallelism

## Why This Matters

FSDP shards model parameters but reconstructs full layers for computation. Tensor parallelism goes further: it splits individual matrix operations across GPUs so each GPU only ever sees a shard of the weight matrix. This is how Megatron-LM trains 100B+ parameter models. The math of how column-parallel and row-parallel linear layers compose without extra communication is elegant and non-obvious -- it comes up in virtually every LLM infrastructure interview.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. What Tensor Parallelism Solves

Consider a transformer FFN layer with shape (d_model=8192, d_ff=32768). In float16:
- Weight matrix: 8192 * 32768 * 2 bytes = **512 MB per layer**
- A 96-layer model: 96 * 512 MB = **48 GB just for FFN weights**

FSDP shards these across N GPUs, but must gather them back before computation. Tensor parallelism splits the **computation** itself:
- GPU 0 computes `x @ W[:, :d_ff//4]` (first quarter of output features)
- GPU 1 computes `x @ W[:, d_ff//4:d_ff//2]` (second quarter)
- ...

Each GPU holds a different weight shard AND computes on it -- no gather needed for the forward pass. This reduces both memory AND communication compared to FSDP for very large layers.

In [ ]:
# Column-parallel linear: split output features across GPUs
# Simulating what Megatron-LM does

def column_parallel_linear(x, W_shards):
    """
    Each 'GPU' (shard) computes x @ W_i where W_i is a column slice of W.
    Result: each GPU holds a different slice of the output.
    No communication needed for this operation!
    """
    return [x @ W for W in W_shards]  # [shard0_output, shard1_output, ...]

def row_parallel_linear(x_shards, W_shards):
    """
    Each GPU holds a column slice of input x and corresponding row slice of W.
    Computes partial sums, then all-reduce to get the full output.
    """
    partial_outputs = [xs @ W for xs, W in zip(x_shards, W_shards)]
    # All-reduce: sum partial outputs (simulated)
    full_output = sum(partial_outputs)
    return full_output  # same result on all GPUs after all-reduce

# Verify correctness
torch.manual_seed(42)
batch_size, d_in, d_out = 4, 8, 16
n_shards = 4  # simulating 4 GPUs

x = torch.randn(batch_size, d_in)
W_full = torch.randn(d_in, d_out)

# Ground truth
out_true = x @ W_full

# Tensor parallel: column split the weight
W_shards = W_full.chunk(n_shards, dim=1)  # split output features
col_outputs = column_parallel_linear(x, W_shards)

print(f"Input: {x.shape}")
print(f"Full weight: {W_full.shape}")
print(f"Each GPU's weight shard: {W_shards[0].shape}")
print(f"Each GPU's output shard: {col_outputs[0].shape}")
print()

# Verify: concatenating shards gives the same as full matmul
out_tp = torch.cat(col_outputs, dim=1)
print(f"Full output match: {torch.allclose(out_true, out_tp, atol=1e-5)}")
print(f"True output[0]: {out_true[0, :4].tolist()}")
print(f"TP output[0]:   {out_tp[0, :4].tolist()}")

## 2. The Column-Parallel + Row-Parallel Fusion

The key insight from Megatron-LM: you can fuse two operations to eliminate an intermediate all-reduce.

For a two-layer FFN (`x -> GeLU(x @ W1 + b1) @ W2 + b2`):

**Naive approach** (requires 2 all-reduces):
```
[column-parallel W1] -> [all-reduce] -> [apply GeLU] -> [column-parallel W2] -> [all-reduce]
```

**Megatron approach** (requires only 1 all-reduce):
```
[column-parallel W1] -> [apply GeLU on shard] -> [row-parallel W2] -> [all-reduce]
```

Why this works: column-parallel leaves output shards on each GPU. GeLU is elementwise, so it can be applied to the shard independently. Row-parallel then consumes the shards directly, producing partial sums that are all-reduced once at the end.

This halves the communication cost of the FFN layer.

In [ ]:
# Megatron-style fused FFN: column-parallel + row-parallel with 1 all-reduce

def gelu(x):
    return x * 0.5 * (1.0 + torch.erf(x / (2.0 ** 0.5)))

def megatron_ffn(x, W1_shards, b1_shards, W2_shards, b2):
    """
    FFN with tensor parallelism following Megatron-LM pattern.
    Only 1 all-reduce needed (at the end).
    
    x: (batch, d_model) -- same on all GPUs
    W1_shards: list of (d_model, d_ff/N) per GPU
    b1_shards: list of (d_ff/N,) per GPU
    W2_shards: list of (d_ff/N, d_model) per GPU
    b2: (d_model,) -- same on all GPUs (added after all-reduce)
    """
    partial_outputs = []
    for W1, b1, W2 in zip(W1_shards, b1_shards, W2_shards):
        # Column-parallel: each GPU computes its output feature shard
        h = gelu(x @ W1 + b1)        # (batch, d_ff/N) -- local
        # Row-parallel: each GPU computes partial sum of final output
        out_partial = h @ W2          # (batch, d_model) -- partial sum
        partial_outputs.append(out_partial)
    
    # All-reduce: sum partial outputs (1 communication round)
    out = sum(partial_outputs) + b2
    return out

# Verify correctness against reference
torch.manual_seed(0)
batch_size, d_model, d_ff = 4, 16, 32
n_gpus = 4

# Reference (single-GPU)
W1 = torch.randn(d_model, d_ff)
b1 = torch.zeros(d_ff)
W2 = torch.randn(d_ff, d_model)
b2 = torch.zeros(d_model)
x = torch.randn(batch_size, d_model)

ref_out = gelu(x @ W1 + b1) @ W2 + b2

# Tensor-parallel shards
W1_shards = W1.chunk(n_gpus, dim=1)
b1_shards = b1.chunk(n_gpus, dim=0)
W2_shards = W2.chunk(n_gpus, dim=0)

tp_out = megatron_ffn(x, W1_shards, b1_shards, W2_shards, b2)

print(f"FFN correctness: {torch.allclose(ref_out, tp_out, atol=1e-5)}")
print(f"Reference output[0]: {ref_out[0, :4].tolist()}")
print(f"TP output[0]:        {tp_out[0, :4].tolist()}")
print(f"\nCommunication: 1 all-reduce instead of 2")
print(f"Memory per GPU: {d_model*d_ff/n_gpus/d_model*100:.0f}% of full FFN weights")

## 3. Attention Head Parallelism

Self-attention can also be parallelized. Multi-head attention naturally decomposes: each GPU handles a subset of attention heads.

For attention with H heads split across N GPUs:
- Each GPU holds `H/N` heads' worth of Q/K/V projections
- Each GPU computes attention for its `H/N` heads independently
- The per-head attention is a column-parallel operation on the output projection

The output projection `W_o` is row-parallel: each GPU has the rows corresponding to its heads, and the final output requires an all-reduce.

Key insight: attention is naturally parallel because heads don't interact with each other until the output projection.

In [ ]:
# Multi-head attention with tensor parallelism
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)
    attn = torch.softmax(scores, dim=-1)
    return attn @ V

def tensor_parallel_mha(x, Wq_shards, Wk_shards, Wv_shards, Wo_shards, n_heads_per_shard):
    """
    Multi-head attention with head parallelism.
    Each GPU handles n_heads_per_shard attention heads.
    """
    batch, seq_len, d_model = x.shape
    d_head = d_model // (n_heads_per_shard * len(Wq_shards))
    
    partial_outputs = []
    for Wq, Wk, Wv, Wo in zip(Wq_shards, Wk_shards, Wv_shards, Wo_shards):
        # Project to Q/K/V for this GPU's heads
        Q = x @ Wq  # (batch, seq, d_head * n_heads_per_shard)
        K = x @ Wk
        V = x @ Wv
        
        # Reshape to separate heads
        Q = Q.view(batch, seq_len, n_heads_per_shard, d_head).transpose(1, 2)
        K = K.view(batch, seq_len, n_heads_per_shard, d_head).transpose(1, 2)
        V = V.view(batch, seq_len, n_heads_per_shard, d_head).transpose(1, 2)
        
        # Attention for this GPU's heads
        attn_out = scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        
        # Output projection shard
        partial_outputs.append(attn_out @ Wo)  # row-parallel
    
    # All-reduce: sum partial outputs from all GPUs
    return sum(partial_outputs)

# Verify
torch.manual_seed(1)
batch, seq_len, d_model = 2, 8, 32
n_heads = 4
n_gpus = 2
n_heads_per_shard = n_heads // n_gpus
d_head = d_model // n_heads

x = torch.randn(batch, seq_len, d_model)

Wq = torch.randn(d_model, d_model) * 0.02
Wk = torch.randn(d_model, d_model) * 0.02
Wv = torch.randn(d_model, d_model) * 0.02
Wo = torch.randn(d_model, d_model) * 0.02

# Reference
Q = x @ Wq
K = x @ Wk
V = x @ Wv
Q = Q.view(batch, seq_len, n_heads, d_head).transpose(1, 2)
K = K.view(batch, seq_len, n_heads, d_head).transpose(1, 2)
V = V.view(batch, seq_len, n_heads, d_head).transpose(1, 2)
attn = scaled_dot_product_attention(Q, K, V)
ref_out = attn.transpose(1, 2).contiguous().view(batch, seq_len, d_model) @ Wo

# Tensor parallel: split heads across GPUs
Wq_shards = Wq.chunk(n_gpus, dim=1)
Wk_shards = Wk.chunk(n_gpus, dim=1)
Wv_shards = Wv.chunk(n_gpus, dim=1)
Wo_shards = Wo.chunk(n_gpus, dim=0)

tp_out = tensor_parallel_mha(x, Wq_shards, Wk_shards, Wv_shards, Wo_shards, n_heads_per_shard)

print(f"MHA correctness: {torch.allclose(ref_out, tp_out, atol=1e-4)}")
print(f"Input: {x.shape}, Output: {tp_out.shape}")
print(f"Communication: 1 all-reduce per attention layer")
print(f"Head allocation: {n_heads_per_shard} heads per GPU")

## 4. Tensor Parallelism Degree: Choosing N

Larger tensor-parallel degree N:
- Reduces per-GPU memory proportionally
- Requires more all-reduces within each layer
- Works best when GPUs communicate via NVLink (high bandwidth, low latency)

Rule of thumb for transformer layers:

| TP Degree | Recommended setup | Bandwidth requirement |
|-----------|-------------------|----------------------|
| 1 (no TP) | Single node with FSDP | Any |
| 2-4 | Small node, NVLink within pair | NVLink |
| 8 | Full NVLink mesh (A100 DGX) | NVLink required |
| > 8 | Multi-node (needs InfiniBand) | 200+ GB/s IB |

**The key constraint**: tensor parallelism all-reduces happen at every layer, in the critical path. InfiniBand (200 GB/s) at TP=16 means each layer pays ~10ms for a 2 GB all-reduce. For a 96-layer model, this is ~1 second per forward pass just in communication.

In [ ]:
# Communication analysis for different TP degrees
def tp_comm_cost_ms(d_model, d_ff, seq_len, batch, tp_degree, bandwidth_gbs):
    """
    Estimate communication time per forward+backward pass.
    Two all-reduces per FFN (but Megatron fuses to 1).
    Two all-reduces per attention layer.
    => 2 all-reduces per transformer layer total.
    """
    # Each all-reduce size: (batch, seq, d_model) in fp16
    tensor_bytes = batch * seq_len * d_model * 2  # bfloat16
    # Ring all-reduce: 2*(N-1)/N * size per GPU
    allreduce_bytes = 2 * (tp_degree - 1) / tp_degree * tensor_bytes
    allreduce_time_ms = allreduce_bytes / (bandwidth_gbs * 1e9) * 1000
    return allreduce_time_ms  # per layer, per pass

# Model params
d_model, d_ff, seq_len, batch = 8192, 32768, 2048, 4
n_layers = 96

nvlink_bw = 600   # GB/s, NVLink 3.0
ib_bw = 200        # GB/s, InfiniBand HDR

print(f"Model: d_model={d_model}, seq_len={seq_len}, batch={batch}, n_layers={n_layers}")
print()
print(f"{'TP Degree':<12} {'Comm/layer (NVLink)':<24} {'Comm/layer (IB)':<20} {'Total (96L, NVLink)':<22}")
print("-" * 80)

for tp in [1, 2, 4, 8, 16, 32]:
    if tp == 1:
        nvlink_ms = 0.0
        ib_ms = 0.0
    else:
        nvlink_ms = tp_comm_cost_ms(d_model, d_ff, seq_len, batch, tp, nvlink_bw)
        ib_ms = tp_comm_cost_ms(d_model, d_ff, seq_len, batch, tp, ib_bw)
    total_ms = nvlink_ms * n_layers * 2  # 2 = fwd + bwd
    print(f"{tp:<12} {nvlink_ms:<24.3f} {ib_ms:<20.3f} {total_ms:<22.1f}")

print()
print("Key insight: TP > 8 on InfiniBand adds seconds of communication overhead per step.")
print("Use TP=8 max within a node (NVLink), combine with Pipeline Parallelism across nodes.")

## Summary

### Key Concepts

| Technique | Splits | Communication | Best for |
|-----------|--------|---------------|---------|
| Column-parallel | Output features of W | None during forward | First of two fused layers |
| Row-parallel | Input features of W | 1 all-reduce per layer | Last of two fused layers |
| Head parallelism | Attention heads | 1 all-reduce per layer | Multi-head attention |
| Combined FFN | Both W1 and W2 | 1 all-reduce (fused!) | Full transformer layer |

**The Megatron trick**: column-parallel W1 + row-parallel W2 in FFN requires only **1 all-reduce** instead of 2. This cuts communication cost in half.

**Practical TP limits**:
- TP=8: ideal on a single 8-GPU NVLink node
- TP=16+: requires crossing node boundaries via InfiniBand -- usually not worth it
- Combine TP (within node) + PP (across nodes) for 100B+ parameter models

**Next**: `08_pipeline_parallelism.ipynb` -- splitting model layers across GPUs for ultra-large models.